# Part 9 (실습) — create_agent로 리리마켓 상담 에이전트 만들기

> **이 노트북의 목표**
> Part 7의 도구 + Part 8의 RAG를 합쳐 실제 작동하는 상담 에이전트를 완성함. `create_agent`로 도구·RAG·system_prompt를 결합하고, 복합 질문에 여러 도구를 골라 쓰는 과정을 messages로 추적함.
>
> **사용 모델**: 채팅 `gemini-3-flash`, 임베딩 `gemini-embedding-001` · **선행**: Part 0~8 (중급의 결합)

## 0. 준비

In [ ]:
%pip install -q -U langchain langchain-google-genai langchain-text-splitters

In [ ]:
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.agents import create_agent
from langchain_core.tools import tool
model = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
print("준비 완료")

## 1. 일반 도구 준비 (Part 7)

재고 조회, 환율, 배송 추적 도구를 만듦. docstring에 "언제 쓰는지" 명시.

In [ ]:
@tool
def get_stock(product: str) -> str:
    """상품명으로 현재 재고 수량을 조회한다. 재고가 있는지 물을 때 사용한다."""
    db = {"베이지 니트": 12, "트렌치 코트": 0, "린넨 셔츠": 5}
    return f"{product} 재고: {db.get(product, '정보 없음')}"

@tool
def get_exchange_rate(currency: str) -> str:
    """원화 대비 환율을 조회한다(USD, EUR 등). 해외 가격을 원화로 환산할 때 사용한다."""
    rates = {"USD": 1380, "EUR": 1490}
    return f"1 {currency} = {rates.get(currency, '정보 없음')}원"

@tool
def track_delivery(order_id: str) -> str:
    """주문번호로 배송 상태를 조회한다. 배송이 어디쯤인지 물을 때 사용한다."""
    status = {"A123": "배송 중 (수도권)", "A124": "배송 완료"}
    return f"주문 {order_id}: {status.get(order_id, '정보 없음')}"

print("일반 도구 3개 준비 완료")

## 2. RAG를 도구로 (Part 8)

정책 문서로 RAG 체인을 만들고, `@tool`로 감싸 "필요할 때만 검색하는" 도구로 등록함.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

policy = """리리마켓 정책.
반품: 수령 후 7일 이내 가능, 단순 변심은 왕복 배송비 고객 부담.
교환: 14일 이내 동일 상품 사이즈/색상 변경 가능, 재고 없으면 환불.
환불: 반품 도착 확인 후 3영업일 이내 결제수단으로 처리.
반품 불가: 착용 흔적·택 제거 상품, 세일 특가 상품.
배송: 평일 오후 2시 이전 주문 당일 출고, 도서산간 1~2일 추가."""

chunks = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=15).split_text(policy)
store = InMemoryVectorStore.from_texts(chunks, embedding=embeddings)
retriever = store.as_retriever(search_kwargs={"k": 2})

rag_prompt = ChatPromptTemplate.from_template(
    "문서를 근거로만 답해. 없으면 '확인이 어렵다'고 해.\n[문서]\n{context}\n[질문]\n{question}"
)
rag_chain = (
    {"context": retriever | (lambda ds: "\n".join(d.page_content for d in ds)),
     "question": RunnablePassthrough()}
    | rag_prompt | model | StrOutputParser()
)

@tool
def search_policy(question: str) -> str:
    """리리마켓의 반품·교환·환불·배송 정책을 검색해 답한다.
    고객이 정책·규정·기간을 물을 때 사용한다."""
    return rag_chain.invoke(question)

print("RAG 도구(search_policy) 준비 완료")

## 3. 에이전트 조립 — create_agent

일반 도구 + RAG 도구 + system_prompt(행동 지침)를 결합함.

In [ ]:
agent = create_agent(
    model=model,
    tools=[get_stock, get_exchange_rate, track_delivery, search_policy],
    system_prompt=(
        "너는 리리마켓의 친절한 상담원이야. 20~40대 여성 고객을 응대해.\n"
        "- 정책·규정·기간 질문은 반드시 search_policy 도구로 확인하고 답해.\n"
        "- 재고는 get_stock, 배송은 track_delivery, 환율은 get_exchange_rate를 써.\n"
        "- 모르면 추측하지 말고 확인이 어렵다고 안내해.\n"
        "- 답은 친절하고 간결하게."
    ),
)
print("리리마켓 상담 에이전트 완성!")

## 4. 실행하고 흐름 추적하기

messages를 펼쳐 "어떤 도구를 왜 불렀는지" 추적하는 헬퍼를 만듦.

In [ ]:
def ask(agent, question):
    print(f"🙋 사용자: {question}")
    result = agent.invoke({"messages": [("user", question)]})
    for m in result["messages"]:
        if getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                print(f"   🔧 도구 호출: {tc['name']}({tc['args']})")
        elif type(m).__name__ == "ToolMessage":
            print(f"   👁 도구 결과: {m.content[:50]}")
    print(f"🤖 답변: {result['messages'][-1].content}\n")
    return result

# 재고 질문 → get_stock을 골라야 함
ask(agent, "트렌치 코트 재고 있어요?");

## 5. 질문마다 다른 도구를 고른다

같은 에이전트가 질문 성격에 따라 다른 도구를 고름.

In [ ]:
ask(agent, "반품은 며칠 안에 해야 하나요?");   # → search_policy (RAG)
ask(agent, "주문 A123 배송 어디쯤이에요?");      # → track_delivery
ask(agent, "50달러는 원화로 얼마예요?");          # → get_exchange_rate

> 📌 정책 질문엔 RAG 도구(search_policy), 배송엔 track_delivery... system_prompt의 가이드 + 각 도구의 docstring이 함께 작동해 정확히 고름.

## 6. 복합 질문 — 여러 도구를 순차로

한 질문에 여러 정보가 필요하면, 에이전트가 루프를 돌며 도구를 여러 번 호출함.

In [ ]:
ask(agent, "트렌치 코트 재고 확인하고, 반품 기간도 알려줘요");
# → get_stock + search_policy 를 모두 호출

## 7. 도구가 필요 없거나, 모르는 질문

In [ ]:
ask(agent, "안녕하세요!");                       # 도구 없이 인사
ask(agent, "리리마켓 VIP 등급은 몇 단계예요?");   # 정책 문서에 없음 → "확인 어렵다"

## ✏️ 미니 실습

**과제**: 위 에이전트에 "쿠폰 코드를 받아 할인율을 알려주는" 도구 `check_coupon`을 추가하고, "WELCOME10 쿠폰 얼마 할인돼요?"라고 물어보기.

In [ ]:
# TODO: 직접 작성
# @tool
# def check_coupon(code: str) -> str:
#     """쿠폰 코드의 할인율을 조회한다. 쿠폰/할인을 물을 때 사용한다."""
#     coupons = {"WELCOME10": "10% 할인", "AUTUMN20": "20% 할인"}
#     return ...
# agent2 = create_agent(model=model, tools=[...기존..., check_coupon], system_prompt=...)
# ask(agent2, "WELCOME10 쿠폰 얼마 할인돼요?")

<details><summary>👉 정답</summary>

```python
@tool
def check_coupon(code: str) -> str:
    """쿠폰 코드의 할인율을 조회한다. 쿠폰/할인 코드를 물을 때 사용한다."""
    coupons = {"WELCOME10": "10% 할인", "AUTUMN20": "20% 할인"}
    return f"{code}: {coupons.get(code, '유효하지 않은 코드')}"

agent2 = create_agent(
    model=model,
    tools=[get_stock, get_exchange_rate, track_delivery, search_policy, check_coupon],
    system_prompt="너는 리리마켓 상담원이야. 쿠폰 질문은 check_coupon을 써.",
)
ask(agent2, "WELCOME10 쿠폰 얼마 할인돼요?")
```
</details>

## 정리

| 절 | 한 일 | 핵심 |
|---|---|---|
| 1~2 | 도구 + RAG 도구 준비 | RAG를 @tool로 감쌈 |
| 3 | create_agent 조립 | model+tools+system_prompt |
| 4~5 | 도구 선택 추적 | 질문마다 다른 도구 |
| 6 | 복합 질문 | 여러 도구 순차 호출 |
| 7 | 한계 | 도구 불필요 / 모르는 질문 |

### 3줄 요약
1. `create_agent(model, tools, system_prompt)`로 도구·RAG를 쥔 에이전트를 완성함.
2. RAG를 @tool로 감싸면 "필요할 때만 검색"하는 동적 판단이 가능해짐.
3. messages를 추적하면 도구 호출 흐름이 다 보임(기본 디버깅).

### ▶ 다음 (Part 10)
지금 에이전트는 대화를 기억 못 함("아까 그 상품"을 못 이어감). **메모리와 상태**를 더해 멀티턴 대화를 기억하게 만듦.